# PII NER: сравнение трёх моделей
Обучаем на `redmadrobot-rnd/pii_train`, тестируем на `redmadrobot-rnd/pii_benchmark`.

Модели:
- `DeepPavlov/rubert-base-cased` (RuBERT)
- `xlm-roberta-base`
- `distilbert-base-multilingual-cased`

In [1]:
import json
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset, Dataset
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score
from transformers import (AutoModelForTokenClassification, AutoTokenizer,
                          DataCollatorForTokenClassification, Trainer, TrainingArguments)

MODELS = {
    "rubert": "DeepPavlov/rubert-base-cased",
    "xlm-roberta": "xlm-roberta-base",
    "distilbert-multi": "distilbert-base-multilingual-cased",
}
MAX_LEN = 128
EPOCHS = 2
BATCH = 32
TRAIN_SIZE = 8000  # None — весь train
SEED = 42

## Данные
`tokens` и `ner_tags` хранятся как JSON-строки — парсим в списки.

In [2]:
def to_ds(name, split):
    df = load_dataset(name, split=split).to_pandas()
    for c in ("tokens", "ner_tags"):
        df[c] = df[c].apply(lambda x: json.loads(x) if isinstance(x, str) else list(x))
    return Dataset.from_pandas(df[["tokens", "ner_tags"]])

train_full = to_ds("redmadrobot-rnd/pii_train", "train")
test_ds = to_ds("redmadrobot-rnd/pii_benchmark", "test")

train_ds = train_full.shuffle(seed=SEED)
if TRAIN_SIZE:
    train_ds = train_ds.select(range(min(TRAIN_SIZE, len(train_ds))))

labels = sorted({t for tags in train_full["ner_tags"] for t in tags}, key=lambda x: (x != "O", x))
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}
print(len(train_ds), len(test_ds), len(labels))

8000 2841 43


## Токенизация
Метка ставится на первый сабтокен слова, остальные — `-100`.

In [3]:
def make_tokenize(tok):
    def fn(batch):
        enc = tok(batch["tokens"], is_split_into_words=True, truncation=True, max_length=MAX_LEN)
        enc["labels"] = []
        for i, tags in enumerate(batch["ner_tags"]):
            prev, lab = None, []
            for w in enc.word_ids(i):
                lab.append(-100 if w is None or w == prev else label2id[tags[w]])
                prev = w
            enc["labels"].append(lab)
        return enc
    return fn

## Обучение и оценка

In [ ]:
def run(name, ckpt):
    tok = AutoTokenizer.from_pretrained(ckpt)
    fn = make_tokenize(tok)
    tr, te = [d.map(fn, batched=True, remove_columns=d.column_names) for d in (train_ds, test_ds)]

    model = AutoModelForTokenClassification.from_pretrained(
        ckpt, num_labels=len(labels), id2label=id2label, label2id=label2id)

    args = TrainingArguments(
        output_dir=f"out/{name}", num_train_epochs=EPOCHS, learning_rate=5e-5,
        per_device_train_batch_size=BATCH, per_device_eval_batch_size=BATCH * 2,
        weight_decay=0.01, eval_strategy="no", save_strategy="no",
        fp16=torch.cuda.is_available(), seed=SEED, report_to="none")

    trainer = Trainer(model=model, args=args, train_dataset=tr,
                      data_collator=DataCollatorForTokenClassification(tok))
    trainer.train()

    out = trainer.predict(te)
    gold = out.label_ids
    pred = out.predictions.argmax(-1)
    y_true = [[id2label[i] for i in g[g != -100]] for g in gold]
    y_pred = [[id2label[i] for i, gi in zip(p, g) if gi != -100] for p, g in zip(pred, gold)]
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    return {"precision": precision_score(y_true, y_pred), "recall": recall_score(y_true, y_pred),
            "f1": f1_score(y_true, y_pred)}, report

results, reports = {}, {}
for name, ckpt in MODELS.items():
    results[name], reports[name] = run(name, ckpt)
    print(name, results[name])

## Сравнение моделей (benchmark, entity-level)

In [5]:
pd.DataFrame(results).T.round(4).sort_values("f1", ascending=False)

,precision,recall,f1
rubert,0.8796,0.9005,0.8899
xlm-roberta,0.8762,0.8870,0.8816
distilbert-multi,0.8392,0.8507,0.8449


In [6]:
# F1 по типам сущностей
per_class = pd.DataFrame({n: {k: v["f1-score"] for k, v in r.items() if k not in ("micro avg", "macro avg", "weighted avg")}
                          for n, r in reports.items()})
per_class.round(3)

,rubert,xlm-roberta,distilbert-multi
BIRTH_CERTIFICATE,0.901,0.891,0.891
CITY,0.813,0.799,0.781
COUNTRY,0.856,0.821,0.814
CREDIT_CARD,0.920,0.890,0.903
DISTRICT,0.831,0.831,0.765
DRIVER_LICENSE,0.884,0.861,0.796
EMAIL,0.845,0.954,0.898
FIRST_NAME,0.955,0.932,0.902
HOUSE,0.851,0.894,0.784
INN,0.843,0.885,0.824


## Примеры предсказаний (distilbert-multi)

In [11]:
import gc
from seqeval.metrics.sequence_labeling import get_entities
from transformers import PreTrainedModel

NAME = "distilbert-multi"

# модель не возвращалась из run() — ищем её в памяти ядра
model = None
for o in gc.get_objects():
    try:
        if (isinstance(o, PreTrainedModel) and type(o).__name__.endswith("ForTokenClassification")
                and o.config._name_or_path == MODELS[NAME]):
            model = o.eval()
            break
    except ReferenceError:
        pass
tok = AutoTokenizer.from_pretrained(MODELS[NAME])

def predict(words):
    enc = tok(words, is_split_into_words=True, truncation=True, max_length=MAX_LEN, return_tensors="pt").to(model.device)
    with torch.no_grad():
        ids = model(**enc).logits.argmax(-1)[0].tolist()
    tags, prev = [], None
    for j, w in enumerate(enc.word_ids()):
        if w is not None and w != prev:
            tags.append(id2label[ids[j]])
        prev = w
    return tags + ["O"] * (len(words) - len(tags))

def entities(words, tags):
    return [f"{t}: {' '.join(words[s:e + 1])}" for t, s, e in get_entities(tags)]

def show(words, gold=None):
    print("ТЕКСТ:", " ".join(words))
    if gold:
        print("  gold:", entities(words, gold))
    print("  pred:", entities(words, predict(words)))
    print()

for i in test_ds.shuffle(seed=1).select(range(5)):
    show(i["tokens"], i["ner_tags"])

show("Меня зовут Иван Сергеевич Петров , живу в Казани на улице Баумана 12 , телефон +7 912 345-67-89 , почта ivan.petrov@mail.ru".split())

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_19968\3821806166.py:11: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  if (isinstance(o, PreTrainedModel) and type(o).__name__.endswith("ForTokenClassification")


ТЕКСТ: Рассказ о природе Байкала
  gold: []
  pred: ['REGION: Байкала']

ТЕКСТ: Подробнее: http://google . com/user/876
  gold: ['URL: http://google . com/user/876']
  pred: ['URL: http://google . com/user/876']

ТЕКСТ: Для оформления полиса ОМС потребуется паспорт серии 4503 номер 123456 и ваш СНИЛС : 123 - 456 - 789 01 . Также нужен оригинал свидетельства о рождении ( серия II - КЗ , номер 987654 ) .
  gold: ['PASSPORT: 4503', 'PASSPORT: 123456', 'SNILS: 123 - 456 - 789 01', 'BIRTH_CERTIFICATE: II - КЗ', 'BIRTH_CERTIFICATE: 987654']
  pred: ['PASSPORT: 4503', 'PASSPORT: 123456', 'SNILS: 123 - 456 - 789 01', 'BIRTH_CERTIFICATE: II - КЗ', 'BIRTH_CERTIFICATE: 987654']

ТЕКСТ: Создай рекламный текст для сайта https://blog . minzdrav . gov . ru/terms .
  gold: ['URL: https://blog . minzdrav . gov . ru/terms']
  pred: ['URL: https://blog . minzdrav . gov . ru/terms']

ТЕКСТ: передай Леонтьевна ЧТО шеф ищет
  gold: ['MIDDLE_NAME: Леонтьевна']
  pred: ['FIRST_NAME: передай', 'MIDDLE_NAME: Ле